# SpaceX Falcon 9 — Interactive Visual Analytics with Folium

**Objective:** locate every launch site on an interactive map, mark the success/failed launches of each site, and calculate the distances between a launch site and its proximities (coastline, railway, highway and city).

In [1]:
import piplite
await piplite.install(['numpy', 'pandas', 'folium'])

In [2]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians

In [3]:
from js import fetch
import io, json

async def get_text(url):
    resp = await fetch(url)
    if not resp.ok:
        raise Exception(f"HTTP {resp.status} for {url}")
    return await resp.text()

async def get_json(url):
    return json.loads(await get_text(url))

async def get_csv(url):
    return pd.read_csv(io.StringIO(await get_text(url)))

In [4]:
spacex_df = await get_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv")
spacex_df.head()

,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


## Task 1: Mark all launch sites on a map

In [5]:
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [6]:
# NASA Johnson Space Center as the start location
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

for _, site in launch_sites_df.iterrows():
    coordinate = [site['Lat'], site['Long']]
    folium.Circle(coordinate, radius=1000, color='#d35400', fill=True,
                  popup=site['Launch Site']).add_to(site_map)
    folium.map.Marker(
        coordinate,
        icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
                     html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % site['Launch Site'])
    ).add_to(site_map)

site_map.save('map_launch_sites.html')
site_map

All launch sites are close to the Equator line (lower latitude = extra speed from Earth's rotation) and very close to the coast (safety: failed launches fall into the ocean).

## Task 2: Mark the success/failed launches for each site on the map

In [7]:
spacex_df['marker_color'] = spacex_df['class'].apply(lambda c: 'green' if c == 1 else 'red')
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


In [8]:
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
marker_cluster = MarkerCluster().add_to(site_map)

for _, record in spacex_df.iterrows():
    folium.Marker(
        location=[record['Lat'], record['Long']],
        icon=folium.Icon(color='white', icon_color=record['marker_color']),
        popup=f"{record['Launch Site']} - {'Success' if record['class'] == 1 else 'Failure'}"
    ).add_to(marker_cluster)

site_map.save('map_launch_outcomes.html')
site_map

## Task 3: Calculate the distances between a launch site and its proximities

In [9]:
def calculate_distance(lat1, lon1, lat2, lon2):
    """Haversine distance in km"""
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

In [10]:
# Mouse position: hover over the map to read the coordinates of any point
site = launch_sites_df[launch_sites_df['Launch Site'] == 'CCAFS SLC-40'].iloc[0]
launch_site_lat, launch_site_lon = site['Lat'], site['Long']

site_map = folium.Map(location=[launch_site_lat, launch_site_lon], zoom_start=14)
MousePosition(position='topright', separator=' Long: ', prefix='Lat:',
              lat_formatter="function(num) {return L.Util.formatNum(num, 5);};",
              lng_formatter="function(num) {return L.Util.formatNum(num, 5);};").add_to(site_map)
folium.Circle([launch_site_lat, launch_site_lon], radius=200, color='#d35400', fill=True,
              popup='CCAFS SLC-40').add_to(site_map)

# Proximity points read with MousePosition (approximate coordinates)
proximities = {
    'Coastline':                (28.56367, -80.56794),
    'Railway':                  (28.57206, -80.58525),
    'Highway (Samuel C Phillips Pkwy)': (28.56335, -80.57085),
    'City (Titusville)':        (28.61200, -80.80788),
}
colors = {'Coastline': 'blue', 'Railway': 'darkred', 'Highway (Samuel C Phillips Pkwy)': 'green', 'City (Titusville)': 'purple'}

rows = []
for name, (lat, lon) in proximities.items():
    distance = calculate_distance(launch_site_lat, launch_site_lon, lat, lon)
    rows.append({'Proximity': name, 'Distance (km)': round(distance, 2)})
    folium.Marker(
        [lat, lon],
        icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
                     html='<div style="font-size: 12; color:%s;"><b>%s: %.2f KM</b></div>' % (colors[name], name, distance))
    ).add_to(site_map)
    folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], [lat, lon]],
                    weight=2, color=colors[name]).add_to(site_map)

site_map.fit_bounds([[launch_site_lat, launch_site_lon]] + [list(p) for p in proximities.values()])
site_map.save('map_proximities.html')
display(pd.DataFrame(rows))
site_map

,Proximity,Distance (km)
0,Coastline,0.87
1,Railway,1.28
2,Highway (Samuel C Phillips Pkwy),0.58
3,City (Titusville),23.21


**Findings:**
- Launch sites are very close to the **coastline** (< 1 km), so failed launches fall into the ocean.
- They are close to **railways and highways** (~1–2 km) for transporting heavy equipment and personnel.
- They keep a safe distance from **cities** (> 20 km) to protect the population.

## Author

Jhoselyn Vera